# UFLD stage 2: train the PPO patch selector

Trains the PPO agent that selects the next patch, on top of the frozen pretrained Transformer from `0-pretrain.ipynb`. Conformal calibration and evaluation are in `2-CP_inference_cls.ipynb`.

In [ ]:
import os
import sys

# Run from the repository root so that `src/`, `data/` and `models/` resolve correctly
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [ ]:
import torch

from src.config import MODEL_CONFIGS, classifier_checkpoint, ppo_model_name
from src.models.patch_selection.classifier import CausalTransformerClassifierKVCached
from src.models.patch_selection.environment import WaferMapGrid
from src.models.patch_selection.ppo import PPOAgent
from src.trainer.train_ppo import training_loop_ppo_transformer
from src.utils.dataloader import load_mixedwm38_splits, get_loaders
from src.utils.utils import seed_everything

## Configuration

In [ ]:
SEED = 4
TRAIN = False          # False: load the checkpoint from models/
MODEL_SIZE = 'large'   # must match the pretrained classifier

DATA_PATH = 'data/MixedWM38.npz'

seed_everything(seed=SEED)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

dataset_name = 'mixedwm38'
num_classes = 8
input_channels = 1
img_size = (52, 52)
batch_size = 128

## Data

In [ ]:
train_dataset, val_dataset, calib_dataset, test_dataset = load_mixedwm38_splits(
    DATA_PATH, img_size=img_size, input_channels=input_channels, num_classes=num_classes)

train_loader, test_loader, val_loader, calib_loader = get_loaders(
    train_dataset, test_dataset, val_dataset=val_dataset, calib_dataset=calib_dataset, batch_size=batch_size)

## Pretrained classifier

In [ ]:
embedding_dim, num_layers = MODEL_CONFIGS[MODEL_SIZE]
hidden_dim = 2 * embedding_dim
num_heads = 4
patch_size = 4
num_patches = (img_size[0] // patch_size) ** 2   # 169

classifier = CausalTransformerClassifierKVCached(
    input_dim=patch_size ** 2,
    hidden_dim=hidden_dim,
    num_classes=num_classes,
    seq_len=num_patches,
    num_layers=num_layers,
    num_heads=num_heads,
    patch_size=patch_size,
    dropout_prob=0.1,
    batch_first=True,
    layer_norm=True,
    use_cls_token=False,
    embedding_dim=embedding_dim,
    use_patch_id_emb=True,
)
classifier.load_weights(classifier_checkpoint(dataset_name, SEED, num_layers, hidden_dim, num_heads=num_heads))
classifier = classifier.to(device)
classifier.eval()
classifier.freeze_all()

## PPO agent

In [ ]:
# The agent is trained on D3 (val_loader); test_loader is used to select the best checkpoint
env = WaferMapGrid(classifier, val_loader, test_loader, start_policy=0, reward_fun="sparse", device=device)

n_epochs = 100
lr, gamma, gae_lambda = 1e-3, 0.99, 0.99

agent = PPOAgent(n_actions=env.action_space.n, obs_dim=classifier.embed_dim,
                 batch_size=batch_size, gamma=gamma, alpha=lr, gae_lambda=gae_lambda,
                 n_epochs=n_epochs, policy_clip=0.2, seed=SEED, chkpt_dir='models',
                 model_name=ppo_model_name(dataset_name, num_layers, hidden_dim, num_heads=num_heads),
                 device=device)

if TRAIN:
    agent = training_loop_ppo_transformer(agent, env, n_epochs=n_epochs)

agent.load_models()

num_params = sum(p.numel() for p in agent.actor.parameters()) + sum(p.numel() for p in agent.critic.parameters())
print(f"Trainable parameters (actor + critic): {num_params:,}")